# Fine-tune a Local ResearchAssist Model with Soup

This notebook is a portfolio-friendly workflow for fine-tuning a small chat model with [Soup](https://github.com/MakazhanAlpamys/Soup), serving it through an OpenAI-compatible local API, and pointing ResearchAssist at that local model.

Soup's README describes the core flow as `pip install "soup-cli[train]"`, `soup init --template chat`, and `soup train`; it also documents `soup serve --model ./output` as an OpenAI-compatible server. Use this notebook as the repeatable experiment record for that flow.

## Portfolio Scope

- Base project: ResearchAssist RAG + LangGraph multi-agent app.
- Training target: supervised instruction tuning for research-assistant tone, grounding, and cautious claims.
- Runtime target: user-owned local model served on the laptop/workstation, wired through `LLM_PROVIDER=local`.
- What to improve later: larger dataset, eval set, model-card notes, and before/after answer comparisons.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "finetune" / "researchassist_sft_sample.jsonl"
SOUP_CONFIG_PATH = PROJECT_ROOT / "notebooks" / "soup_researchassist.yaml"
OUTPUT_DIR = PROJECT_ROOT / "models" / "researchassist-soup"

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Soup config:", SOUP_CONFIG_PATH)
print("Output:", OUTPUT_DIR)

## 1. Install Soup

Run this in a Python 3.10-3.12 environment. GPU is recommended for useful training; CPU is mainly for smoke testing.

In [ ]:
# Uncomment when running the notebook in a fresh environment.
# %pip install "soup-cli[train,serve]"

In [ ]:
!soup doctor

## 2. Inspect the Starter Dataset

Replace this sample with 100-1,000+ high-quality examples for a meaningful pet project. Good examples can come from manually curated paper QA, critique rewrites, summary preferences, and failure cases where the agent should admit missing context.

In [ ]:
import json

with DATA_PATH.open() as f:
    rows = [json.loads(line) for line in f]

print(f"Rows: {len(rows)}")
rows[0]

## 3. Write a Soup Config

The default base model is intentionally small enough for local experimentation. Change `base` to a stronger model when your machine can handle it.

In [ ]:
config = f"""
base: Qwen/Qwen2.5-0.5B-Instruct
task: sft

data:
  train: {DATA_PATH.as_posix()}
  format: alpaca
  val_split: 0.2

training:
  epochs: 1
  lr: 2e-5
  batch_size: auto
  lora:
    r: 16
    alpha: 16
  quantization: 4bit
  stream_layers: true

output: {OUTPUT_DIR.as_posix()}
""".strip()

SOUP_CONFIG_PATH.write_text(config + "\n")
print(SOUP_CONFIG_PATH.read_text())

## 4. Train

This is the expensive cell. For a resume-worthy project, keep the config, dataset version, hardware notes, and final metrics in git or an experiment log.

In [ ]:
!soup train --config "{SOUP_CONFIG_PATH}"

## 5. Smoke Test the Fine-tuned Model

In [ ]:
!soup chat --model "{OUTPUT_DIR}"

## 6. Serve Locally for ResearchAssist

Run this in a terminal and leave it open:

```bash
soup serve --model ./models/researchassist-soup --host 127.0.0.1 --port 8001
```

Then set ResearchAssist `.env`:

```bash
LLM_PROVIDER=local
LOCAL_LLM_BASE_URL=http://127.0.0.1:8001/v1
LOCAL_LLM_MODEL=researchassist-soup
```

For Docker Compose, use `LOCAL_LLM_BASE_URL=http://host.docker.internal:8001/v1` so the backend container reaches the model server running on the host.

In [ ]:
import requests

base_url = "http://127.0.0.1:8001/v1"
response = requests.get(f"{base_url}/models", timeout=5)
print(response.status_code)
print(response.text[:1000])

## 7. Evaluate Before Shipping

Minimum checks for this pet project:

- Ask 10 fixed research QA prompts against OpenRouter and the local model.
- Score grounding, helpfulness, refusal to invent missing context, and latency.
- Keep failure examples and add them back into `data/finetune/` after manual cleanup.
- Confirm `/api/model` reports `provider=local`, the expected model name, and `status=connected`.